In [37]:
import sys
import os
from pathlib import Path

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [38]:
from helper import RAGHelper

from src.utils.helper import get_questions_table_name, get_eval_table_name, get_questions_table_name, get_vector_db

from src.utils.helper import *


In [39]:
import psycopg
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_groq import ChatGroq
import uuid
import re
from tenacity import retry, retry_if_exception_type, stop_after_attempt
from tenacity.wait import wait_base
load_dotenv()


True

In [40]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.12.0+cpu
None
False


In [41]:
class wait_for_rate_limit(wait_base):
    def __init__(self, fallback_wait=60):
        self.fallback_wait = fallback_wait

    def __call__(self, retry_state):
        exception = retry_state.outcome.exception()
        if exception:
            error_msg = str(exception)
            
            # Handle Groq's specific format: "try again in 13m19.199s"
            match = re.search(r'try again in (?:(\d+)m)?([\d.]+)s', error_msg)
            if match:
                minutes = int(match.group(1)) if match.group(1) else 0
                seconds = float(match.group(2))
                wait_time = (minutes * 60) + seconds + 2.0 
                print(f"\n⏳ Groq Rate Limit! Pausing execution for {wait_time:.1f} seconds...")
                return wait_time
            
            # Handle OpenRouter/general rate limit errors
            # Look for rate limit keywords and use a reasonable default wait time
            if "rate limit" in error_msg.lower() or "RateLimitError" in error_msg:
                print(f"\n⏳ Rate limit detected. Pausing execution for {self.fallback_wait} seconds...")
                return self.fallback_wait
                
        # If it's a different error, fallback to default wait time
        return self.fallback_wait

In [42]:
@retry(
    retry=retry_if_exception_type(Exception), 
    wait=wait_for_rate_limit(fallback_wait=60),
    stop=stop_after_attempt(5)
)
def process_single_question(q, session_id, rg, expert_domain, retriever, gen_model, eval_model,
                             chunk_size, chunk_overlap, eval_table, database_url, rag_db, retrieval_type):
    # Extract data from the tuple
    qid = q[0]
    query = q[1]
    
    print(f"Processing QID {qid}...")
    
    # 1. Generate Response
    query, response, sent_list = rg.simple_rag(
        query=query, 
        expert_domain=expert_domain, 
        retriever=retriever,
        gen_model=gen_model
    )
    
    # 2. Evaluate
    eval_response = rg.evaluate_rag(query, response, sent_list, eval_model=eval_model)
    
    # 3. Calculate Metrics
    rg.get_metrics(eval_response, sent_list)
    
    # 4. Insert into Database
    rg.db_insert(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        table_name=eval_table, 
        database_url=database_url,
        qid=qid, 
        vector_db=rag_db, 
        session_id=session_id,
        retrieval_type=retrieval_type
    )
    print(f"✅ Successfully inserted QID {qid}")

In [43]:
# Use a shared local cache so Hugging Face models are not re-downloaded every run
HF_CACHE_DIR = Path(parent_dir) / ".cache" / "huggingface"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("TRANSFORMERS_CACHE", str(HF_CACHE_DIR / "transformers"))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE_DIR / "hub"))

# Reuse embedding instances across the loop to avoid reloading the same model
EMBEDDING_CACHE = {}

def build_embedding_function(model: str) -> HuggingFaceEmbeddings:
    if model not in EMBEDDING_CACHE:
        print(f"Loading embedding model: {model}")
        EMBEDDING_CACHE[model] = HuggingFaceEmbeddings(
            model=model,
            # model_kwargs={
            #     "device": "cuda"
            # },
            # encode_kwargs={
            #     "batch_size": 64,
            #     "normalize_embeddings": True,
            # },
            cache_folder=str(HF_CACHE_DIR),
        )
    else:
        print(f"Reusing embedding model from cache: {model}")
    return EMBEDDING_CACHE[model]


def build_retriever(embedder: str, embedding_fn, chromadb_folder: str, db_name: str,
                     retrieval_type: str = "dense", search_kwargs: dict = None):
    search_kwargs = search_kwargs or {"k": 3}
    collection_name = embedder.replace("/", "_") + "_cs"
    persist_directory = f"{chromadb_folder}/{db_name}"
    vector_db = Chroma(collection_name=collection_name, embedding_function=embedding_fn, persist_directory=persist_directory)
    return vector_db.as_retriever(retrieval_type=retrieval_type, search_kwargs=search_kwargs)


def fetch_pending_questions(database_url: str, questions_table: str, limit: int = 1):
    conn = psycopg.connect(database_url)
    cursor = conn.cursor()
    rows = cursor.execute(f"select a.id, a.query from {questions_table} as a limit {limit}").fetchall()
    conn.close()
    return rows

In [44]:
def run_experiment(*, database_url, questions_table, eval_table, chromadb_folder, db_name,
                    embedder, expert_domain, provider, api_key, gen_model, eval_model,
                    rag_db="Chroma", retrieval_type="dense", search_kwargs=None,
                    question_limit=1):
    chunk_size = int(db_name.split("_")[1])
    chunk_overlap = int(db_name.split("_")[2])

    embedding_fn = build_embedding_function(embedder)
    retriever = build_retriever(embedder, embedding_fn, chromadb_folder, db_name,
                                 retrieval_type=retrieval_type, search_kwargs=search_kwargs)

    rg = RAGHelper(api_key=api_key, provider=provider)

    questions = fetch_pending_questions(database_url, questions_table, limit=question_limit)

    session_id = uuid.uuid4()
    for q in questions:
        process_single_question(q, session_id, rg, expert_domain, retriever, gen_model, eval_model,
                                 chunk_size, chunk_overlap, eval_table, database_url, rag_db, retrieval_type)

    print("🎉 All questions processed and evaluated!")
    return rg, retriever, questions

In [45]:
DATABASE_URL = os.getenv("DATABASE_URL")
# questions_table = "cs_questions" 
# eval_table = "cs_eval_v3"
# retrieval_type = 'dense'
# search_kwargs = {"k":3}

# rag_db = "Chroma"
# chromadb_folder = "../database"
# db_name = "customer support_256_50"
# embedder =  "BAAI/LLM-Embedder" #"BAAI/bge-base-en-v1.5" 
# expert_domain = "customer support"

# # ## Groq
# # provider = "groq"
# # gen_model = "llama-3.1-8b-instant"
# # eval_model = "openai/gpt-oss-20b"#"llama-3.3-70b-versatile"#"openai/gpt-oss-120b"

# # ## LM Studio
# # provider = "lmstudio"
# # gen_model = "nvidia-nemotron-nano-12b-v2-vl-bf16"
# # eval_model = "nvidia-nemotron-nano-12b-v2-vl-bf16"


# ## OpenRouter
# provider = "openrouter"
# gen_model = "nvidia/nemotron-nano-9b-v2:free"
# eval_model = "nvidia/nemotron-3-nano-30b-a3b:free"

In [46]:
# print(retrieval_type)

In [47]:
# print(f"""select a.id, a.query from {questions_table} as a left join {eval_table} as b 
# on a.id = b.id and b.embed_model<> 'BAAI/LLM-Embedder'
# where b.chunk_size is null limit 2""")


In [48]:
# for r in sql_response:
#     print(r)

In [49]:
def loop_domains():   
    
    questions_table = get_questions_table_name(DOMAINS)
    eval_table = get_eval_table_name(DOMAINS)    

     # Loop through domains and datasets
    for domain_short_name, data_set in DOMAINS.items():

        print(f"Processing domain: {domain_short_name}")        
        # Loop through datasets
        for data_set_path, name in data_set.items():

            print(f"  Processing dataset: {data_set_path} > {name}")
            
            for chunk_size, chunk_overlap in zip(CHUNKING_SIZES, CHUNKING_OVERLAPS):
                print(f"    Processing chunk size: {chunk_size} and overlap: {chunk_overlap}")

                 # Loop Embedder function
                for embedder in EMBEDDING_MODELS:
                    print(f"      Processing embedder: {embedder}")

                    collection_name = get_collection_name(embedder, domain_short_name)

                    # Generate embeddings
                    embedding_function = build_embedding_function(embedder)

                    # Loop Vector Database
                    for vector_db_type in VECTOR_DATABASES:

                        persist_directory = get_persist_directory(vector_db_type, domain_short_name, chunk_size, chunk_overlap)     
                        vector_db = get_vector_db(persist_directory, collection_name, embedding_function)    
                        
                        rg, retriever, sql_response = run_experiment(
                            database_url=DATABASE_URL,
                            questions_table=questions_table,
                            eval_table=eval_table,
                            chromadb_folder=chromadb_folder,
                            db_name=db_name,
                            embedder=embedder,
                            expert_domain=expert_domain,
                            provider=provider,
                            api_key=os.getenv("OPEN_ROUTER_API_KEY"),
                            gen_model=gen_model,
                            eval_model=eval_model,
                            rag_db=rag_db,
                            retrieval_type=retrieval_type,
                            search_kwargs=search_kwargs,
                            question_limit=1,
                        )
                        



In [33]:
loop_domains()

Processing domain: cs
  Processing dataset: delucionqa > Jeep manual
    Processing chunk size: 256 and overlap: 50
      Processing embedder: BAAI/LLM-Embedder
Loading embedding model: BAAI/LLM-Embedder


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

NameError: name 'chromadb_folder' is not defined

In [ ]:
# rg, retriever, sql_response = run_experiment(
#     database_url=DATABASE_URL,
#     questions_table=questions_table,
#     eval_table=eval_table,
#     chromadb_folder=chromadb_folder,
#     db_name=db_name,
#     embedder=embedder,
#     expert_domain=expert_domain,
#     provider=provider,
#     api_key=os.getenv("OPEN_ROUTER_API_KEY"),
#     gen_model=gen_model,
#     eval_model=eval_model,
#     rag_db=rag_db,
#     retrieval_type=retrieval_type,
#     search_kwargs=search_kwargs,
#     question_limit=1,
# )

In [ ]:
session_id = uuid.uuid4()

for q in sql_response:
#     process_single_question(q, session_id)
    
# print("🎉 All questions processed and evaluated!")
    
    query = q[1]
    query, response, sent_list = rg.simple_rag(query=query, expert_domain=expert_domain, retriever=retriever,gen_model=gen_model)
    eval_response = rg.evaluate_rag(query, response, sent_list, eval_model=eval_model)
    print(eval_response)
    relevance, utilization, completeness, adherence = rg.get_metrics(eval_response, sent_list)
    # rg.db_insert( chunk_size=chunk_size,
    #           chunk_overlap=chunk_overlap,
    #           table_name=eval_table, database_url=DATABASE_URL,
    #           qid=q[0], vector_db=rag_db, session_id=session_id,
    #               retrieval_type=retrieval_type)
    print(relevance, utilization, completeness, adherence)
    
    
    # print(f"Query: {query}\n response:{response}")